# Colab GPU connection test
Select Kernel > Select Another Kernel > Colab > New Colab Server > GPU (a free option if offered). Sign in personally when prompted, then Run All and save this local notebook. Auto Connect may select CPU, so verify the output. This notebook only tests compute; it never trains or accesses your dataset. Notebook metadata alone does not allocate a GPU.


In [ ]:
import json, platform, sys
from pathlib import Path
import torch
available = torch.cuda.is_available()
result = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": available,
    "gpu_name": torch.cuda.get_device_name(0) if available else None,
    "execution_location": str(Path.cwd()),
    "is_colab_runtime": Path("/content").is_dir() and "google.colab" in sys.modules,
}
if available:
    x = torch.randn(64, 64, device="cuda")
    result["cuda_tensor_test"] = bool(torch.isfinite(x @ x.T).all().item())
else:
    result["cuda_tensor_test"] = False
print(json.dumps(result, indent=2))
if Path("/content").is_dir():
    Path("/content/colab_gpu_status.json").write_text(json.dumps(result, indent=2))
assert available, "No CUDA GPU: select a Colab GPU server, then run this cell again."


In [ ]:
import json, torch
from pathlib import Path
assert Path('/content').is_dir(), 'The selected kernel is not a Colab runtime.'
assert torch.cuda.is_available(), 'CUDA GPU is unavailable.'
x=torch.randn(64,64,device='cuda')
print(json.dumps({'probe':'weed-supported-vscode-api-001','gpu':torch.cuda.get_device_name(0),'cuda_ok':bool(torch.isfinite(x@x.T).all().item()),'cwd':str(Path.cwd())}))


In [ ]:
import hashlib, json, os, sys, subprocess, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), "A CUDA GPU is required; CPU fallback is disabled."
assert Path("/content").is_dir(), "Select the existing Colab GPU kernel."
source_zip=Path("/content/colab_source.zip")
dataset_zip=Path("/content/weed_training_224.zip")
for file, expected in [(source_zip,'3aa9cb751f9f49044cb06adf71d83463827eda8f0a3c995310c959bc1a99ea69'),(dataset_zip,'f89c6341456711fcc6a02ec2d41991cbce6075a0124f561bb91b969512ddc380')]:
    assert file.is_file(), f"Upload {file.name} with the official Colab extension."
    with file.open("rb") as stream:
        assert hashlib.file_digest(stream,"sha256").hexdigest()==expected, f"Transfer hash mismatch: {file}"
project=Path("/content/AI-Weed-Assistant")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(source_zip) as archive:
    for entry in archive.infolist():
        assert (project/entry.filename).resolve().is_relative_to(project.resolve())
    archive.extractall(project)
os.chdir(project)
if str(project) not in sys.path: sys.path.insert(0,str(project))
from training.archive import extract_dataset
data=Path("/content/weed_dataset")
if not (data/"manifest.json").is_file():
    extract_dataset(dataset_zip,data)
manifest=json.loads((data/"manifest.json").read_text())
assert len(manifest["classes"])==12 and len(manifest["records"])==5536, "Dataset coverage mismatch"
for package in ["torchvision","numpy","scipy","sklearn","matplotlib","PIL"]:
    __import__(package)
cfg=json.loads(Path("training/config.json").read_text())
cfg.update(data_dir=str(data),output_dir="/content/weed_outputs/models/mobilenet_v2",evaluation_dir="/content/weed_outputs/evaluation/mobilenet_v2",device="cuda",num_workers=2)
resume=Path(cfg["output_dir"])/"last.pt"
if resume.is_file(): cfg["resume"]=str(resume)
Path("training/colab_config.json").write_text(json.dumps(cfg,indent=2))
print(json.dumps({"gpu":torch.cuda.get_device_name(0),"classes":manifest["classes"],"full_split_counts":{name:sum(r["split"]==name for r in manifest["records"]) for name in ["train","val","test"]},"config":cfg},indent=2))
pid_file=Path("/content/weed_training_pid.json")
existing=None
if pid_file.exists():
    previous=json.loads(pid_file.read_text())
    try:
        os.kill(previous["pid"],0)
        existing=previous["pid"]
    except ProcessLookupError:
        pass
if existing:
    print("Existing training process is still running:",existing)
else:
    outputs=Path("/content/weed_outputs");outputs.mkdir(exist_ok=True)
    log=(outputs/"training.log").open("a",buffering=1)
    process=subprocess.Popen([sys.executable,"-u","-m","training.run_job","--config","training/colab_config.json"],cwd=project,stdout=log,stderr=subprocess.STDOUT)
    pid_file.write_text(json.dumps({"pid":process.pid,"config":cfg},indent=2))
    print("Full-dataset GPU training process started:",process.pid)


In [ ]:
import hashlib, json, os, sys, subprocess, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), "A CUDA GPU is required; CPU fallback is disabled."
assert Path("/content").is_dir(), "Select the existing Colab GPU kernel."
source_zip=Path("/content/colab_source.zip")
dataset_zip=Path("/content/weed_training_224.zip")
for file, expected in [(source_zip,'3aa9cb751f9f49044cb06adf71d83463827eda8f0a3c995310c959bc1a99ea69'),(dataset_zip,'f89c6341456711fcc6a02ec2d41991cbce6075a0124f561bb91b969512ddc380')]:
    assert file.is_file(), f"Upload {file.name} with the official Colab extension."
    with file.open("rb") as stream:
        assert hashlib.file_digest(stream,"sha256").hexdigest()==expected, f"Transfer hash mismatch: {file}"
project=Path("/content/AI-Weed-Assistant")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(source_zip) as archive:
    for entry in archive.infolist():
        assert (project/entry.filename).resolve().is_relative_to(project.resolve())
    archive.extractall(project)
os.chdir(project)
if str(project) not in sys.path: sys.path.insert(0,str(project))
from training.archive import extract_dataset
data=Path("/content/weed_dataset")
if not (data/"manifest.json").is_file():
    extract_dataset(dataset_zip,data)
manifest=json.loads((data/"manifest.json").read_text())
assert len(manifest["classes"])==12 and len(manifest["records"])==5536, "Dataset coverage mismatch"
for package in ["torchvision","numpy","scipy","sklearn","matplotlib","PIL"]:
    __import__(package)
cfg=json.loads(Path("training/config.json").read_text())
cfg.update(data_dir=str(data),output_dir="/content/weed_outputs/models/mobilenet_v2",evaluation_dir="/content/weed_outputs/evaluation/mobilenet_v2",device="cuda",num_workers=2)
resume=Path(cfg["output_dir"])/"last.pt"
if resume.is_file(): cfg["resume"]=str(resume)
Path("training/colab_config.json").write_text(json.dumps(cfg,indent=2))
print(json.dumps({"gpu":torch.cuda.get_device_name(0),"classes":manifest["classes"],"full_split_counts":{name:sum(r["split"]==name for r in manifest["records"]) for name in ["train","val","test"]},"config":cfg},indent=2))
pid_file=Path("/content/weed_training_pid.json")
existing=None
if pid_file.exists():
    previous=json.loads(pid_file.read_text())
    try:
        os.kill(previous["pid"],0)
        existing=previous["pid"]
    except ProcessLookupError:
        pass
if existing:
    print("Existing training process is still running:",existing)
else:
    outputs=Path("/content/weed_outputs");outputs.mkdir(exist_ok=True)
    log=(outputs/"training.log").open("a",buffering=1)
    process=subprocess.Popen([sys.executable,"-u","-m","training.run_job","--config","training/colab_config.json"],cwd=project,stdout=log,stderr=subprocess.STDOUT)
    pid_file.write_text(json.dumps({"pid":process.pid,"config":cfg},indent=2))
    print("Full-dataset GPU training process started:",process.pid)


In [ ]:
import json
from pathlib import Path
print(json.dumps({p.name:p.stat().st_size for p in Path('/content').glob('*') if p.is_file() and (p.name.startswith('weed_') or p.name.endswith('manifest.json') or p.name=='colab_source.zip')},indent=2))


In [ ]:
import hashlib, json, os, sys, subprocess, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), "A CUDA GPU is required; CPU fallback is disabled."
assert Path("/content").is_dir(), "Select the existing Colab GPU kernel."
source_zip=Path("/content/colab_source.zip")
dataset_zip=Path("/content/weed_training_224.zip")
if not dataset_zip.is_file():
    transfer=json.loads(Path("/content/transfer_manifest.json").read_text())
    temporary=dataset_zip.with_suffix(".joining")
    with temporary.open("wb") as outgoing:
        for part in transfer["parts"]:
            piece=Path("/content")/part["name"]
            assert piece.is_file(), f"Missing transfer part: {piece.name}"
            content=piece.read_bytes()
            assert hashlib.sha256(content).hexdigest()==part["sha256"], f"Transfer part hash mismatch: {piece.name}"
            outgoing.write(content)
    temporary.replace(dataset_zip)
for file, expected in [(source_zip,'8ef54983a996e10695db4192ac91289f64fe6317568759ef26e3bff7270a6594')]:
    assert file.is_file(), f"Upload {file.name} with the official Colab extension."
    with file.open("rb") as stream:
        assert hashlib.file_digest(stream,"sha256").hexdigest()==expected, f"Transfer hash mismatch: {file}"
with dataset_zip.open("rb") as stream:
    image_archive_hash=hashlib.file_digest(stream,"sha256").hexdigest()
assert image_archive_hash in {'2e203e434649cb21800908f607f96b3582e8aefc892d416ba81903b5fc948c06','f89c6341456711fcc6a02ec2d41991cbce6075a0124f561bb91b969512ddc380'}, "Dataset transfer SHA256 mismatch"
project=Path("/content/AI-Weed-Assistant")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(source_zip) as archive:
    for entry in archive.infolist():
        assert (project/entry.filename).resolve().is_relative_to(project.resolve())
    archive.extractall(project)
os.chdir(project)
if str(project) not in sys.path: sys.path.insert(0,str(project))
from training.prepare_runtime import prepare_runtime_dataset
data=Path("/content/weed_dataset")
final_metadata=Path("/content/final_splits.json.gz")
if image_archive_hash!='2e203e434649cb21800908f607f96b3582e8aefc892d416ba81903b5fc948c06':
    assert final_metadata.is_file(), "Upload final_splits.json.gz to apply final group-aware assignments"
    assert hashlib.sha256(final_metadata.read_bytes()).hexdigest()=='40040419ff9df9a2f19da71fcb9f48dba652791dc409ca23ca9e88bbe887c1fa', "Final split manifest hash mismatch"
    manifest=prepare_runtime_dataset(dataset_zip,data,final_metadata)
else:
    manifest=prepare_runtime_dataset(dataset_zip,data)
assert len(manifest["classes"])==12 and len(manifest["records"])==5536, "Dataset coverage mismatch"
for package in ["torchvision","numpy","scipy","sklearn","matplotlib","PIL"]:
    __import__(package)
cfg=json.loads(Path("training/config.json").read_text())
cfg.update(data_dir=str(data),output_dir="/content/weed_outputs/models/mobilenet_v2",evaluation_dir="/content/weed_outputs/evaluation/mobilenet_v2",device="cuda",num_workers=2)
resume=Path(cfg["output_dir"])/"last.pt"
if resume.is_file(): cfg["resume"]=str(resume)
Path("training/colab_config.json").write_text(json.dumps(cfg,indent=2))
print(json.dumps({"gpu":torch.cuda.get_device_name(0),"classes":manifest["classes"],"full_split_counts":{name:sum(r["split"]==name for r in manifest["records"]) for name in ["train","val","test"]},"config":cfg},indent=2))
pid_file=Path("/content/weed_training_pid.json")
existing=None
if pid_file.exists():
    previous=json.loads(pid_file.read_text())
    try:
        os.kill(previous["pid"],0)
        command=Path(f"/proc/{previous['pid']}/cmdline").read_bytes()
        if b"training.run_job" in command: existing=previous["pid"]
    except ProcessLookupError:
        pass
if existing:
    print("Existing training process is still running:",existing)
else:
    outputs=Path("/content/weed_outputs");outputs.mkdir(exist_ok=True)
    log=(outputs/"training.log").open("a",buffering=1)
    process=subprocess.Popen([sys.executable,"-u","-m","training.run_job","--config","training/colab_config.json"],cwd=project,stdout=log,stderr=subprocess.STDOUT)
    pid_file.write_text(json.dumps({"pid":process.pid,"config":cfg},indent=2))
    print("Full-dataset GPU training process started:",process.pid)


In [ ]:
import json, subprocess
from pathlib import Path
job=Path('/content/weed_outputs/job_status.json')
progress=Path('/content/weed_outputs/models/mobilenet_v2/training_status.json')
log=Path('/content/weed_outputs/training.log')
result={'job':json.loads(job.read_text()) if job.exists() else None,'progress':json.loads(progress.read_text()) if progress.exists() else None,'log_tail':log.read_text(errors='replace')[-2500:] if log.exists() else None}
result['gpu_status']=subprocess.run(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()
print(json.dumps(result,indent=2))
